# Polynomial regression

In [ ]:
import warnings
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

## Data that follows a curve: y = 0.8x² + 0.9x + 2 + noise

In [ ]:
rng = np.random.default_rng(42)
X = 6 * rng.random((200, 1)) - 3
y = (0.8 * X ** 2 + 0.9 * X + 2 + rng.standard_normal((200, 1))).ravel()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
lr = LinearRegression().fit(X_train, y_train)
print("straight line, test R2", round(r2_score(y_test, lr.predict(X_test)), 3))

## Add an x² column

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=True)
X_train_p = poly.fit_transform(X_train)
X_test_p = poly.transform(X_test)
print(X_train[0], "->", X_train_p[0].round(3))     # [x] -> [1, x, x^2]
lr2 = LinearRegression().fit(X_train_p, y_train)
print("degree 2, test R2", round(r2_score(y_test, lr2.predict(X_test_p)), 3))
print("coefficients", lr2.coef_.round(3), "intercept", round(lr2.intercept_, 3))   # close to 0.9, 0.8 and 2

## Interactive: drag the slider to change the degree (25 training points)

In [ ]:
Xs = 6 * rng.random((25, 1)) - 3
ys = (0.8 * Xs ** 2 + 0.9 * Xs + 2 + rng.standard_normal((25, 1))).ravel()
grid = np.linspace(-3, 3, 300).reshape(-1, 1)
degrees = list(range(1, 16))
fig = go.Figure([go.Scatter(x=Xs.ravel(), y=ys, mode="markers", name="training points")])
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression()).fit(Xs, ys)
    fig.add_scatter(x=grid.ravel(), y=np.clip(m.predict(grid), -5, 15), mode="lines", name=f"degree {d}", visible=(d == 2))
steps = [{"label": str(d), "method": "update", "args": [{"visible": [True] + [dd == d for dd in degrees]}]} for d in degrees]
fig.update_layout(sliders=[{"steps": steps, "active": 1, "currentvalue": {"prefix": "degree: "}}], yaxis_range=[-5, 15])
fig

## Two inputs: the new columns include x², y² and xy

In [ ]:
p = PolynomialFeatures(degree=2).fit(np.zeros((1, 2)))
print(p.get_feature_names_out(["x", "y"]))
print("degree 30 with 2 inputs gives", PolynomialFeatures(30).fit(np.zeros((1, 2))).n_output_features_, "columns")

In [ ]:
x = 7 * rng.random(100) - 2.8
yy = 7 * rng.random(100) - 2.8
z = x ** 2 + yy ** 2 + 0.2 * x + 0.2 * yy + 0.1 * x * yy + 2 + rng.standard_normal(100)
XY = np.c_[x, yy]                    # two columns side by side (not np.array([x, y]).reshape(100, 2))
print("plane R2   ", round(LinearRegression().fit(XY, z).score(XY, z), 3))
print("degree 2 R2", round(make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(XY, z).score(XY, z), 3))